###  A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
!pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [38]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [39]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')

if api_key and api_key.startswith('sk-or-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'openai/gpt-5-nano'
openai = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

API key looks good so far


In [40]:
link=fetch_website_links('https://edwarddonner.com')
link

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

###  First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [41]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [42]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [43]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

### select_relevant_links with improvements to function

In [46]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [47]:
select_relevant_links('https://edwarddonner.com')

Selecting relevant links for https://edwarddonner.com by calling openai/gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [48]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-5-nano
Found 15 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'pricing page - endpoints anchor',
   'url': 'https://huggingface.co/pricing#endpoints'},
  {'type': 'pricing page - spaces anchor',
   'url': 'https://huggingface.co/pricing#spaces'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discord', 'url': 'https://huggi

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [49]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [50]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling openai/gpt-5-nano
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.6-35B-A3B
Updated
6 days ago
•
335k
•
1.06k
tencent/HY-Embodied-0.5
Updated
7 days ago
•
1.66k
•
888
unsloth/Qwen3.6-35B-A3B-GGUF
Updated
about 17 hours ago
•
816k
•
574
baidu/ERNIE-Image
Updated
4 days ago
•
4.14k
•
501
tencent/HY-World-2.0
Updated
5 days ago
•
500
Browse 2M+ models
Spaces
Running
on
Zero
Agents
Featured
634
OmniVoice
🌍
634
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
2.26k
Wan2.2 14B Preview
🐌
2.26k
generate a video from an image with a text prompt
Running
144
Bonsai 1-bi

In [51]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [52]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [53]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-5-nano
Found 12 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.6-35B-A3B\nUpdated\n6 days ago\n•\n335k\n•\n1.07k\ntencent/HY-Embodied-0.5\nUpdated\n7 days ago\n•\n1.66k\n•\n888\nunsloth/Qwen3.6-35B-A3B-GGUF\nUpdated\nabout 17 hours ago\n•\n816k\n•\n574\nbaidu/ERNIE-Image\nUpdated\n4 days ago\n•\n4.14k\n•\n501\ntencent/HY-World-2.0\nUpdated\n5 days ago\n•\n501\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nAgents\nFeatured\n634\nOmniVoice\

In [23]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [24]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [25]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-5-nano
Found 12 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.6-35B-A3B\nUpdated\n5 days ago\n•\n335k\n•\n991\ntencent/HY-Embodied-0.5\nUpdated\n6 days ago\n•\n1.66k\n•\n881\nunsloth/Qwen3.6-35B-A3B-GGUF\nUpdated\nabout 5 hours ago\n•\n816k\n•\n531\nbaidu/ERNIE-Image\nUpdated\n3 days ago\n•\n4.14k\n•\n484\ntencent/HY-World-2.0\nUpdated\n4 days ago\n•\n479\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nAgents\nFeatured\n621\nOmniVoice\n🌍\

In [54]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [55]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-5-nano
Found 8 relevant links


# Hugging Face Brochure

## About Hugging Face  
Hugging Face is the AI community building the future of machine learning (ML). It serves as the central collaboration platform for ML engineers, scientists, and enthusiasts worldwide to create, discover, and share models, datasets, and applications. With over 2 million models, 500,000+ datasets, and 1 million+ applications hosted, Hugging Face fosters an open and ethical AI future.

## Platform Features  
- **Model & Dataset Hosting:** Host and collaborate on unlimited public models and datasets spanning various modalities — text, image, video, audio, and 3D.  
- **Spaces:** Run and share live ML apps and demos, enabling easy interaction with models without setup.  
- **Open Source Stack:** Accelerate ML development with Hugging Face’s open source tools.  
- **Enterprise Solutions:** Paid compute and enterprise-grade solutions for teams to scale and deploy ML efficiently.  
- **Portfolio Building:** Share your ML projects and build a professional portfolio within the community.

## Community & Collaboration  
Hugging Face is truly a community-led platform where members actively collaborate to improve AI technologies. It empowers the next generation of ML practitioners through transparency and open exchange, with features designed to promote easy sharing and discovery.

## Company Culture  
- Emphasizes openness, collaboration, and ethical AI development.  
- Fosters a passionate community focused on innovation and learning.  
- Supports diversity and inclusivity in the AI ecosystem.  
- Encourages contributors to build and share their work publicly to drive collective progress.

## Customers and Users  
- Individual ML engineers and data scientists looking for state-of-the-art models and datasets.  
- AI startups and companies requiring scalable ML infrastructure and deployment tools.  
- Researchers and academics advancing machine learning and AI.  
- Enterprises seeking customized solutions and compute resources for their AI projects.

## Careers  
While specific job listings are not included here, Hugging Face values talented individuals passionate about machine learning, AI ethics, open source, and community building. Careers likely focus on roles in:

- Machine Learning Engineering and Research  
- Software Development and Infrastructure  
- Community and Developer Relations  
- Product Management and Enterprise Solutions  

Candidates interested in shaping the future of AI in a collaborative, innovative environment are encouraged to explore opportunities via the Hugging Face website.

---

**Join Hugging Face today to be part of an open AI future — where community, collaboration, and cutting-edge ML intersect!**

Website: [huggingface.co](https://huggingface.co)  
Contact: Accessible via the website for enterprise and partnership inquiries.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [56]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [57]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the leading AI community platform dedicated to building the future of machine learning. It serves as a vibrant hub where the global machine learning community collaborates by sharing models, datasets, and applications. The platform hosts over 2 million models and offers access to more than 500,000 datasets, enabling developers and researchers to accelerate AI innovation.

**Mission:** To empower machine learning practitioners and organizations with cutting-edge tools and resources in an open, collaborative environment.

---

## What Hugging Face Offers

### Platform Highlights

- **Models:** Access and browse over 2 million machine learning models spanning text, image, video, audio, and even 3D modalities.
- **Datasets:** Explore and collaborate on 500k+ datasets that fuel AI research and applications.
- **Spaces:** Run and share AI applications on a user-friendly platform supporting a wide variety of ML applications.
- **Buckets:** Efficiently manage and store large-scale datasets and models.
- **Open Source Stack:** Hugging Face provides an open-source ecosystem to help build, train, and deploy ML models faster.
- **Community Collaboration:** Share your work, build your ML portfolio, and connect with like-minded AI specialists worldwide.

### Accelerate Your AI Projects

- Access to **paid Compute and Enterprise solutions** designed to scale your AI efforts.
- High-quality voice cloning, image editing, video generation, and other specialized ML apps hosted on Spaces.

---

## Enterprise & Team Solutions

Hugging Face offers robust plans designed to empower organizations of any size:

### Team Plan – Starting at $20/user/month

- Advanced AI platform with **enterprise-grade security**.
- **Single Sign-On (SSO)** integration for secure access.
- Granular access via **Resource Groups** and **Token Management**.
- Comprehensive **audit logs** and **analytics** dashboards.
- Private storage with **1 TB per user** plus options for additional storage.
- **Priority support** from the Hugging Face expert team.
- Scalable **compute options**, including ZeroGPU Quota Boost (5x more quota).
- Manage budgets effectively with **managed billing and yearly commitments**.

### Enterprise Plan – Custom Pricing (Contact Sales)

- Fully flexible contract options tailored to organizational needs.
- Additional features such as **organization-wide security policies**.
- Dedicated support, advanced billing controls, and scalable infrastructure.
- Ideal for forward-thinking AI enterprises aiming to integrate Hugging Face tools at scale.

---

## Pricing Overview

| Plan       | Features                                                                                     | Price                       |
|------------|----------------------------------------------------------------------------------------------|-----------------------------|
| **PRO**    | Boost personal use; 10x private storage; 20x inference credits; Spaces hosting; PRO badge.    | $9/month                    |
| **Team**   | Team collaboration tools; enterprise security; analytics; priority support.                  | $20 per user/month          |
| **Enterprise** | Custom plans with flexible contracts, enhanced security, and support.                      | Contact sales for pricing   |

---

## Company Culture

- **Community-Centric:** Hugging Face is deeply embedded in an open and collaborative machine learning ecosystem.
- **Innovation-Driven:** Encourages rapid experimentation and sharing of new ML models and technologies.
- **Inclusive and Diverse:** Supports a broad international community of AI practitioners, researchers, and developers.
- **Supportive Environment:** Provides dedicated enterprise support and resources to help teams thrive.

---

## Customers & Partners

Hugging Face powers AI collaboration for:

- Leading AI researchers and practitioners worldwide.
- Innovative startups building ML-powered applications.
- Large enterprises looking for scalable, secure AI solutions.
- Academic institutions leveraging curated datasets and models.

Organizations that value cutting-edge, community-driven AI development rely on Hugging Face to scale and secure their machine learning initiatives.

---

## Careers at Hugging Face

Hugging Face welcomes talented individuals passionate about AI and open-source technology. The company encourages applicants who want to:

- Work at the intersection of AI research, software engineering, and community building.
- Contribute to impactful projects supporting millions of ML practitioners globally.
- Be part of an energetic, forward-looking team committed to shaping the future of AI.

Check the Hugging Face website for current job openings and internship opportunities.

---

## Get Started Today

- Join over a million members of the AI community collaborating on Hugging Face.
- Explore millions of ML models, datasets, and applications.
- Scale your AI initiatives with secure enterprise-grade tools.
- Share your AI projects and build your professional machine learning profile.

Visit **https://huggingface.co** to sign up and start building the future of AI.

---

**Hugging Face** — The AI community building the future.

In [58]:
stream_brochure("accuratess", "https://accuratess.com/services")

Selecting relevant links for https://accuratess.com/services by calling openai/gpt-5-nano
Found 0 relevant links


# Accurate - Precision Meets Excellence

## About Accurate
Accurate is a leading company dedicated to delivering high-quality, precise solutions designed to meet the evolving needs of our diverse clientele. While our brand name underscores our commitment to accuracy, our core mission extends beyond that to include innovation, reliability, and customer satisfaction.

## Company Culture
At Accurate, we foster a culture of excellence driven by meticulous attention to detail and a passion for innovation. Our team thrives in an environment that encourages continuous learning, collaboration, and a commitment to refining every aspect of our products and services. We value integrity, transparency, and respect, ensuring that both our employees and customers feel supported and valued.

## Our Customers
We serve a wide range of industries that require precise and dependable solutions. Our customer base includes professionals and organizations who demand accuracy and high performance, relying on Accurate to deliver consistent results that empower their own success.

## Careers at Accurate
Join us to be part of a dynamic team where your skills and ideas contribute directly to creating solutions that make a difference. We offer opportunities for growth in a variety of roles centered around innovation, quality assurance, and customer engagement. Accurate is committed to nurturing talent and providing a workplace where your career can flourish.

---

Discover the precision and dedication that define Accurate. Whether you are a potential customer seeking reliable services or a professional looking to advance your career, Accurate is the partner you can trust.